<div align="right" style="text-align: right"><i>Peter Norvig</i></div>

# (How to Write a (Lisp) Interpreter (in Python))

This notebook describes how (and why) to implement computer language interpreters in general, and in particular  an interpreter for most of the [**Scheme**](https://www.scheme.org/) dialect of [**Lisp**](https://en.wikipedia.org/wiki/Lisp_(programming_language%29). I call my language and interpreter **Lispy** because it is Lisp implemented in Python.   

Why should interpreters and compilers matter to you? As [Steve Yegge said](https://steve-yegge.blogspot.com/2007/06/rich-programmer-food.html?), "If you don't know how compilers work, then you don't know how computers work." Yegge describes 8 problems that can be solved with compilers (or equally well with interpreters, or with Yegge's typical heavy dosage of cynicism).

## Syntax and Semantics of  Programs

The **syntax** of a language is the arrangement of characters to form correct statements or expressions. For example, in the language of mathematical expressions (and in many programming languages and handheld calculators), the syntax for computing one plus two is "1 + 2". The **semantics** of a language determines what it means: what computations it describes, and ultimately what answer(s) it computes.  We say that "1 + 2" *evaluates* to 3, and write that as "1 + 2" ⇒ 3. 

If you are familiar with languages such as Python or Java, you may find Scheme syntax to be unusual. Consider:

<table style="border-collapse: collapse; border: 2px solid black;">
  <thead>
    <tr>
      <th style="border: 1px solid black; padding: 10px; text-align: left;">Java</th>
      <th style="border: 1px solid black; padding: 10px; text-align: left;">Lisp</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid black; padding: 10px; vertical-align: top;">
<pre style="margin: 0; font-family: monospace;"><code>if (x.val() &gt; 0) {
    return fn(A[i] + 3 * i,
              new String[] {"one", "two"});
}</code></pre>
      </td>
      <td style="border: 1px solid black; padding: 10px; vertical-align: top;">
<pre style="margin: 0; font-family: monospace;"><code>(if (&gt; (val x) 0)
    (fn (+ (aref A i) (* 3 i))
        (quote (one two))))</code></pre>
      </td>
    </tr>
  </tbody>
</table>              

Java has a wide variety of syntactic conventions (keywords, infix operators, four kinds of brackets, operator precedence, dot notation, quotes, commas, semicolons), but Scheme syntax is much simpler:
Scheme programs consist solely of expressions; there is no statement/expression distinction.
Numbers (e.g. 1) and symbols (e.g. A) are called **atomic expressions**; they cannot be broken into pieces. These are similar to their Java counterparts, except that in Scheme, operators such as `+` and `>` are symbols too, and are treated the same way as `A` and `fn`.
Everything else is a **list expression**: a "(", followed by zero or more expressions, followed by a ")". The first element of the list expression determines what it means:
- A list starting with a keyword, e.g. `(if ...)`, is a **special form**; the meaning depends on the keyword.
- A list starting with a non-keyword, e.g. `(max x y)`, is a function call: the function `max` is applied to the arguments `x` and `y` to compute a value.

The beauty of Scheme is that the full language only needs 5 keywords and 8 syntactic forms. In comparison, Python has 35 keywords and 110 syntactic forms, and Java has 50 keywords and 133 syntactic forms. All those parentheses may seem intimidating, but Scheme syntax has the virtues of simplicity and consistency. (Some have joked that "Lisp" stands for "**L**ots of **I**rritating **S**illy **P**arentheses"; I think it stand for "**L**isp **I**s **S**yntactically **P**ure".)


# Language 1: Lispy Calculator

We won't tackle all of Scheme right away; instead we'll start with a subset of Scheme I call **Lispy Calculator**. Lispy Calculator lets you do any computation you could do on a typical calculator—as long as you are comfortable with prefix notation. And while many calculators let you store and retrieve a value from a fixed set of registers (e.g. A, B, or C), Lispy Calculator let's you define and use any number of variables with any names you choose. Here's an example program that computes the area of a circle of radius 10, using the formula π r<sup>2</sup>

     (begin
       (define r 10)
       (* pi (* r r)))


Here is a table of all the allowable expressions in the Lispy Calculator Language. In the Syntax column of this table, *symbol* must be a symbol, *number* must be an integer or floating point number, and the other italicized words can be any expression. The notation *exp...* means zero or more repetitions of *exp*.

|Expression	|Syntax	|Example|Semantics|
|-----------|-------|-------|-------------|
|constant 	|*number*	|`12` or `-3.45e+6`|A number evaluates to itself.|
|variable |	*symbol*	|`r`|A symbol is interpreted as a variable name; its value is the variable's value.|
|definition	|`(define` *symbol exp*`}`	|`(define r 10)`|Define a new variable and give it the value of the expression *exp*.|
|procedure call	|`(`*proc exp*...`)`	|`(sqrt (* 2 8))` ⇒ 4.0|Proc's value (a function) is applied to the argument values.|


Let's get some imports out of the way, and be explicit about how Scheme objects are represented  in Python:

In [1]:
from numbers import Number
import math
import operator as op

Symbol = str               # A Scheme symbol is implemented as a Python str
Atom   = Symbol | Number   # A Scheme atom is a Symbol or Number
List   = list              # A Scheme list is implemented as a Python list
Exp    = Atom | List       # A Scheme expression is an Atom or List
Env    = dict              # A Scheme environment is a dictionary mapping of {variable: value}

## The core of Lisp: eval 

Here is the core of the interpreter, `eval`. It takes as input an expression, `exp`, and an **environment** that specifies the values of variables. It returns the value of the expression. These few lines are what  what [Alan Kay called](https://queue.acm.org/detail.cfm?id=1039523) "the Maxwell's Equations of Software."


In [2]:
def eval(exp: Exp, env: Env) -> object:
    """Evaluate an expression in an environment."""
    match exp:
        case Number():                             # number evaluates to itself 
            return exp
        case Symbol():                             # variable evaluates to its value in environment
            return env[exp]
        case ('define', Symbol(name), val):                # definition adds name to the environment
            env[name] = eval(val, env)
            return name
        case (proc, *args):                        # procedure call
            func = eval(proc, env)
            vals = [eval(arg, env) for arg in args]
            return func(*vals)

## Global Environment

We mentioned that an **environment** is a mapping from variable names to their values. We will define a default global environment. `ENV`, containing values for the names of a bunch of standard functions like `sqrt` and `max`, as well as operators like `+` and `>`, which are implemented as procedures in Lisp. (Scheme's name for `print` is `display`; the function `unparse` will be defined later.)

In [3]:
ENV = {
    **vars(math), # sqrt, sin, cos, etc.
    '+':op.add, '-':op.sub, '*':op.mul, '/':op.truediv, 
    '>':op.gt,  '<':op.lt,  '>=':op.ge, '<=':op.le, '=':op.eq, 
    'eq?':     op.is_, 'equal?':  op.eq,
    'abs':     abs,
    'append':  op.add,  
    'apply':   lambda proc, args: proc(*args),
    'begin':   lambda *x: x[-1],
    'cons':    lambda x,y: [x] + y,
    'display': lambda x: print(unparse(x)),
    'expt':    pow,
    'first':   lambda x: x[0],
    'length':  len, 
    'list':    lambda *x: List(x), 
    'list?':   lambda x: isinstance(x, list), 
    'map':     lambda f, *args: list(map(f, *args)),
    'max':     max, 
    'min':     min,
    'not':     op.not_,
    'null?':   lambda x: x == [], 
    'number?': lambda x: isinstance(x, Number),  
    'procedure?': callable,
    'rest':    lambda x: x[1:], 
    'round':   round,
    'symbol?': lambda x: isinstance(x, Symbol),
    }

(*Note:* because I am not implementing all the features of Scheme (such as [continuations](https://groups.csail.mit.edu/mac/projects/info/schemedocs/ref-manual/html/scheme_122.html)), I can get away with defining `begin` as a function rather than a special form.)

## Parsing

How do we get from a sequence of characters to the abstract syntax tree that `eval` expects? The function `parse` does the job, in two steps: 
1. **Lexical analysis**: the function `tokenize` breaks the characters into tokens (such as the keyword `"if"` or the number `"10"`).
2. **Syntactic analysis**: the function `parse_tokens` converts the tokens into an expression.

In [4]:
def parse(program: str) -> Exp:
    """Read a Scheme expression from a string.
    First split the program into tokens, then read from the token list."""
    return parse_tokens(tokenize(program))

There are many tools for lexical analysis (such as Mike Lesk and Eric Schmidt's [lex](https://en.wikipedia.org/wiki/Lex_%28software%29)), most of which define tokens as a class containing a token kind and a token string. But Lisp is so simple that there are really only three types of tokens: left paren, right paren, and everything else. So `str.split` can do the job (with a little help):

In [5]:
def tokenize(chars: str) -> list[str]:
    """Convert a string of characters into a list of tokens.
    (Put spaces around parens, then split on spaces.)"""
    return chars.replace('(', ' ( ').replace(')', ' ) ').split()

`parse_tokens` looks at the first token; if it is a `)` that's a syntax error. If it is a `(`, then we start building up a list of sub-expressions until we hit a matching `)`. Any non-parenthesis token must be an atom: first try to interpret it as a number, and failing that, it must be a symbol. 

In [6]:
def parse_tokens(tokens: list[str]) -> Exp:
    """Read an expression from a list of tokens, mutating the list."""
    if not tokens:
        raise SyntaxError('unexpected end of expression')
    match (token := tokens.pop(0)):
        case ')':
            raise SyntaxError('unexpected ")"')
        case '(':
            result = []
            while tokens[0] != ')':
                result.append(parse_tokens(tokens))
            tokens.pop(0) # pop off the closing ')'
            return result
        case _:
            try:
                n = float(token)
                return int(n) if n.is_integer() else n
            except ValueError:
                return token # symbol

## Homoiconicity

One of the defining features of Lisp is [**homoiconicity**](https://en.wikipedia.org/wiki/Homoiconicity), a fancy word derived from the Greek for "same representation" that refers to a language in which the internal representation of a program is the same as the external representation. In other words, a Lisp program is just a list, so all the functions for reading, manipulating and printing lists apply equally to programs. We've already defined `parse` to convert a string into an internal representation; now we'll define `unparse` to reverse the process:

In [7]:
def unparse(exp: Exp) -> str:
    "Convert a Python object back into a Scheme-readable string."
    match exp:
        case List(): return '(' + ' '.join(map(unparse, exp)) + ')'
        case _:      return str(exp)

We can see that parse and unparse are inverses:

In [8]:
unparse(parse("(lambda (A i) (fn (+ (aref A i) (* 3 i))))"))

'(lambda (A i) (fn (+ (aref A i) (* 3 i))))'

Python has a module, `ast` (for "abstract syntax tree") that makes it possible to manipulate programs, but Python is not homoiconic. The internal representation of a program is quite different from the external representation:

In [9]:
import ast
print(ast.dump(ast.parse("lambda A, i: fn(A[i] + 3 * i)"), indent=4))

Module(
    body=[
        Expr(
            value=Lambda(
                args=arguments(
                    args=[
                        arg(arg='A'),
                        arg(arg='i')]),
                body=Call(
                    func=Name(id='fn', ctx=Load()),
                    args=[
                        BinOp(
                            left=Subscript(
                                value=Name(id='A', ctx=Load()),
                                slice=Name(id='i', ctx=Load()),
                                ctx=Load()),
                            op=Add(),
                            right=BinOp(
                                left=Constant(value=3),
                                op=Mult(),
                                right=Name(id='i', ctx=Load())))])))])


## Batch Processing

In **batch processing** an entire program is read, parsed, and evaluated in one step, with no human in the loop.

In [10]:
def batch(program: str) -> None:
    """Parse the program, evaluate it, and print the result."""
    print(unparse(eval(parse(program), ENV)))

Here is a sample program and the result of running it using `batch`:

In [11]:
program1 = """
(begin 
    (define r 10)
    (* pi (* r r))) """

In [12]:
batch(program1)

314.1592653589793


We can also see the intermediate steps of tokenizing and parsing the program:

In [13]:
tokenize(program1)

['(',
 'begin',
 '(',
 'define',
 'r',
 '10',
 ')',
 '(',
 '*',
 'pi',
 '(',
 '*',
 'r',
 'r',
 ')',
 ')',
 ')']

In [14]:
parse(program1)

['begin', ['define', 'r', 10], ['*', 'pi', ['*', 'r', 'r']]]

## Interactive Processing

One of Lisp's great legacies is the notion of an interactive loop: a way for a programmer to enter an expression, see the results, and then think of something new to try. This facilitates exploratory programming: instead of having to design every aspect of a complete program ahead of time, the programmer can experiment, learning as they go, step by step.  So let's define the function `repl` (which stands for read-eval-print-loop):

In [15]:
def repl(prompt='\nlispy> ') -> None:
    """A read-eval-print loop."""
    print('lispy> read-eval-print loop – Type exit to exit')
    while (expr := input(prompt)) != 'exit':
        batch(expr)

Here is an example run of `repl()`:
    
    lispy> read-eval-print loop – Type exit to exit
    lispy> (define r 10)
    r
    
    lispy> (* pi (* r r))
    314.159265359
    
    lispy> (if (> (* 11 11) 120) (* 7 6) oops)
    42
    
    lispy> (list (+ 1 1) (+ 2 2) (* 2 3) (expt 2 3) (expt 2 (expt 2 2)))
    (2 4 6 8 16

    lispy> exit

You can experiment with `repl()` yourself by deleting the `#` and running the following cell. I left it commented out so that when you do the "Run All Cells" command on this notebook, it runs all the way through, without pausing to ask you to type in some Scheme expressions.

In [16]:
#repl()

# Language 2: Full Lispy

We will now extend our language with four new special forms, and one variant of the old `define` special form. This gived us a much more nearly-complete Scheme subset:

|Expression	|Syntax| Example|	Semantics|
|-----------|------|--------|------------|
|conditional	|(`if` *test then_part else_part*`)`|`(if (< x 0) (- x) x)`|	Evaluate *test*; if true, evaluate and return *then part*; otherwise *else part*.|
|quotation	|`(quote` *exp*`)`| `(quote (+ 1 2))` ⇒ `(+ 1 2)`|	Return the exp literally; do not evaluate it.|
|assignment	|`(set!` *symbol exp*`)`| `(set! r2 (* r r))`|	Evaluate *exp* and assign that value to *symbol*.|
|procedure	|`(lambda (`*symbol...*`)` *exp*`)`|`(lambda (r) (* pi (* r r)))`|	Create an anonymous procedure.|
|definition | `(define (`*symbol*...`)` *body*`)`|`(defun (add1 x) (+ x 1)`| Define a named procedure.|

- The `if` special form is similar to  the `(x if test else y)` syntax in Python, although  only `False` counts as false in Scheme.
- The `quote` special form allows you to build a literal data structure.
- The `set!` special form allows you to update the value of a previously-defined variable. This is different from `define`, which introduces a new variable in the curent environment (and sets its initial value).
- The `lambda` special form (an obscure name from Alonzo Church's [lambda calculus](https://en.wikipedia.org/wiki/Lambda_calculus)) creates a procedure (without giving it a name).
- The new option for `define` is just a shortcut for a regular `define` of a `lambda` expression.

There are two equivalent ways of defining a procedure and giving it a name. Consider:

    (define (circle-area r) (* pi (* r r)))

    (define circle-area (lambda (r) (* pi (* r r)))

Either way, `circle-area` is  defined to take as its value a procedure that refers to the global variables `pi` and `*`, and takes a single parameter, `r`. Now we can call the procedure like this:

    (circle-area (+ 5 5))

We want this call to return the value of `(* pi (* r r))` with `r` set to 10. But it wouldn't do to set `r` to 10 in the global environment. What if we were using `r` for some other purpose? Instead, we want to arrange for there to be a **local variable** named `r` that is only used during this call to `circle-area`.  The process for calling a procedure introduces these new local variable(s), binding each symbol in the parameter list of the function to the corresponding value in the argument list of the function call. In this case, the result of the call is 314.159265359.
    

## Local Variables and Procedures

To handle local variables, we will **nest** envronments. Local variables are defined in an environment that is "inside" another environment. We will use the convention: `env['_outer']` to refer to the outer environment of the nested environment `env`. When we evaluate (`circle-area (+ 5 5))`, we will fetch the procedure body, `(* pi (* r r))`, and evaluate it in an environment that has `r` as the sole local variable (with value 10), but also has the global environment as the `_outer` environment; it is there that we will find the values of `*` and `pi`.  In the diagram, the inner environment is blue and the outer red:

<p><table  style="border: 3px solid red" cellspacing=1 cellpadding=5><tr><td>
<tt>pi: 3.141592653589793
<br>*: &lt;built-in function mul&gt;
<br>...
<br>
<table  style="border: 3px solid blue" cellspacing=1 cellpadding=5>
<tr><td>r: 10
</table>
</table>

When we look up a variable in such a nested environment, we look first at the innermost level, but if we don't find the variable name there, we move to the next outer level. 

Here is the definition of the Procedure class:

In [17]:
from dataclasses import dataclass

@dataclass
class Procedure(object):
    """A user-defined Scheme procedure."""
    parms: list[Symbol]
    body:  Exp
    env:   Env
    def __call__(self, *args) -> object: 
        env = Env(zip(self.parms, args), _outer=self.env)
        return eval(self.body, env)
    def __repr__(self) -> str: 
        return f'<Procedure{unparse(self.parms)}>'

We see that a procedure has three components: a list of parameter names, a body expression, and an environment that tells us what other variables are accessible from the body. For a procedure defined at the top level this will be the global environment, but it is also possible for a procedure to refer to the local variables of the environment in which it was defined (**not** the environment in which it is called).

The function `find` is used to find the right environment for a variable: starting with the inner one and going out, find the first environment that mentions the variable name.

In [18]:
def find(var: Symbol, env: Env) -> Env:
    """Find the environment that contains the variable `var`."""
    return env if (var in env) else find(var, env['_outer'])

To see how these all go together, here is the new definition of `eval`:

In [19]:
def eval(exp: Exp, env) -> object:
    """Evaluate an expression in an environment."""
    match exp:
        case Symbol():                             # variable reference
            return find(exp, env)[exp]
        case Number():                             # constant 
            return exp
        case ('if', test, then_, else_):           # conditional evaluates one branch or the other
            branch = (else_ if eval(test, env) is False else then_)
            return eval(branch, env)
        case ('define', (name, *parms), body):     # procedure definition
            env[name] = eval(['lambda', parms, body], env)
            return name
        case ('define', Symbol(name), val):        # regular definition
            env[name] = eval(val, env)
            return name
        case ('quote', constant):                  # constant expression
            return constant
        case ('set!', symbol, val):                # variable assignment
            find(symbol, env)[symbol] = eval(val, env)
            return None
        case ('lambda', parms, body):              # create an anonymous procedure
            return Procedure(parms, body, env)
        case (proc, *args):                        # procedure call
            func = eval(proc, env)
            vals = [eval(arg, env) for arg in args]
            return func(*vals)

For example:

In [20]:
batch("""
(begin 
    (define (circle-area r) (* pi (* r r)))
    (circle-area (+ 5 5)))""")

314.1592653589793


We now have a language with variables, conditionals, sequential execution, and procedures with recursive calls. That makes our language Turing-complete. If you are familiar with other languages, you might think that a `while` or `for` loop would be needed, but Scheme manages to do without these just fine. In Scheme you iterate by defining recursive functions. The Scheme report says "Scheme demonstrates that a very small number of rules for forming expressions, with no restrictions on how they are composed, suffice to form a practical and efficient programming language." 

## How Small/Complete/Good/Fast is Lispy?
    
In which we judge Lispy on several criteria:
- **Small**: Lispy is *very* small: about 120 lines or 4K of source code. (An earlier version was just 90 lines, but was perhaps a bit too terse.)   The smallest version of my Scheme in Java, [Jscheme](http://norvig.com/jscheme.html) was 1664 lines and 57K of source. Jscheme was originally called SILK (Scheme in Fifty Kilobytes), but I only kept under that limit by counting bytecode rather than source code. Lispy does much better; I think it meets Alan Kay's 1972 [claim](http://gagne.homedns.org/~tgagne/contrib/EarlyHistoryST.html) that *you could define the "most powerful language in the world" in "a page of code."* (if you use a small font). However, I think Alan would disagree, because he would count the Python compiler as part of the code, putting me <i>well</i> over a page.
- **Complete**: Lispy is not very complete compared to the [Scheme standard](https://standards.scheme.org/).  Some major shortcomings:
  - **Syntax**: Missing comments, quote and quasiquote notation, # literals, the derived
  expression types (such as `cond` and `let`), and dotted list notation.
  - **Semantics**: Missing `call/cc` and tail recursion.  
  - **Data Types**: Missing strings, characters, booleans, ports,
  vectors, exact/inexact numbers. A Scheme list should actually be a custom data class, not a Python list.
  - **Procedures**: Missing over 100 primitive procedures.
  - **Error recovery**: Lispy does not attempt to detect,
  reasonably report, or recover from errors.  Lispy expects the
  programmer to be perfect. 
- **Good**: That's up to the readers to decide.  I think that Lispy is good for my purpose of explaining Lisp interpreters. It is not a viable choice for serious software development.
- **Fast**: Lispy computes <tt>(factorial 100)</tt> in less than a millisecond. That's fast enough for me. <p>

In [21]:
%%time
batch("""
(begin 
    (define (factorial n) 
       (if (<= n 1) 
           1 
           (* n (factorial (- n 1)))))
    (factorial 100))""")

93326215443944152681699238856266700490715968264381621468592963895217599993229915608941463976156518286253697920827223758251185210916864000000000000000000000000
CPU times: user 395 μs, sys: 200 μs, total: 595 μs
Wall time: 521 μs


## More Example Lisp Programs

In [22]:
batch("""
(list 
    (define (count item L) 
       (if (null? L)
           0
           (+ (equal? item (first L)) (count item (rest L)))))
    (count 0 (list 0 1 2 3 0 0))
    (count (quote the) (quote (the more the merrier the bigger the better))))""")

(count 3 4)


In [23]:
batch("""
(list
    (define (square x) (* x x))
    (define (compose f g) (lambda (x) (f (g x))))
    (define (repeat f) (compose f f))
    ((compose round sqrt) 101)
    ((repeat square) 3)
    ((repeat (repeat square)) 2)
    ((repeat (repeat sqrt)) (pow 2 16)))""")

(square compose repeat 10 81 65536 2.0)


In [24]:
batch("""
(begin
    (define (fib n) 
       (if (< n 2) 
           1 
           (+ (fib (- n 1)) (fib (- n 2)))))
    (define (range start stop)
       (if (= start stop) 
           (quote ()) 
           (cons start (range (+ start 1) stop))))
    (map fib (range 0 20)))""")

(1 1 2 3 5 8 13 21 34 55 89 144 233 377 610 987 1597 2584 4181 6765)


## True Story

To back up the idea that it can be  helpful to know how
interpreters work, here's a story.  Way back in 1984 I was writing a
Ph.D. thesis.  This was before LaTeX, before Microsoft Word for Windows–we used
[troff](https://en.wikipedia.org/wiki/Troff). Unfortunately, troff had no facility for forward references
to symbolic labels: I wanted to be able to write "As we will see on
page @theorem-x" and then write something like "@(set theorem-x \n%)" in
the appropriate place (the troff register \n% holds the page number). My
fellow grad student Tony DeRose felt the same need, and together we
sketched out a simple Lisp program that would handle this as a preprocessor.  However,
it turned out that the Lisp we had at the time was good at reading
Lisp expressions, but slow at reading 100 KB of characters one character at a time.

From there Tony and I split paths.  He reasoned that the hard part was
the interpreter for expressions; he needed Lisp for that, but he knew
how to write a tiny C routine
for reading the characters one at a time, and how to link it into the Lisp
program.  I didn't know how to do that linking, but I reasoned that writing an
interpreter for this trivial language (all it had was set variable,
fetch variable, and string concatenate) was easy, so I wrote an
interpreter in C. So, ironically, Tony wrote a Lisp program (with one small routine in C) because he was a
C programmer, and I wrote a C program (that implements a hand-coded mini-interpreter) because I was a Lisp programmer.

In the end, we both got our theses done (<a href="http://www.eecs.berkeley.edu/Pubs/TechRpts/1985/6081.html">Tony</a>, <a href="http://www.eecs.berkeley.edu/Pubs/TechRpts/1987/5995.html">Peter</a>).

<h2>Further Reading</h2>

Years ago, I showed how to write a semi-practical near-complete Scheme interpreter (one in [Java](https://norvig.com/jscheme.html)  and one in [Common Lisp](https://github.com/norvig/paip-lisp/blob/main/docs/chapter22.md)). I also have another page describing a <a href="http://norvig.com/lispy2.html">more advanced version of Lispy</a>.
   
To learn more about Scheme consult some of the fine books (by
   <a
   href="http://books.google.com/books?id=xyO-KLexVnMC&lpg=PP1&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Friedman
   and Fellesein</a>,
   <a href="http://books.google.com/books?id=wftS4tj4XFMC&lpg=PA300&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Dybvig</a>,
   <a
   href="http://books.google.com/books?id=81mFK8pqh5EC&lpg=PP1&dq=scheme%20programming%20book&pg=PP1#v=onepage&q&f=false">Queinnec</a>,
   <a href="http://www.eecs.berkeley.edu/~bh/ss-toc2.html">Harvey and
   Wright</a> or
   <a href="https://www.researchgate.net/profile/Gerald-Sussman-2/publication/37597721_Structure_and_Interpretation_of_Computer_Programs_H_Abelson_GJ_Sussman_colaboracion_de_J_Sussman_prol_de_Alan_J_Perlis/links/53d141450cf220632f392bf7/Structure-and-Interpretation-of-Computer-Programs-H-Abelson-GJ-Sussman-colaboracion-de-J-Sussman-prol-de-Alan-J-Perlis.pdf">Sussman and Abelson</a>),
   videos (by <a
   href="http://groups.csail.mit.edu/mac/classes/6.001/abelson-sussman-lectures/">Abelson
   and Sussman</a>),
   tutorials (by
      <a
   href="http://www.ccs.neu.edu/home/dorai/t-y-scheme/t-y-scheme.html">Dorai</a>,
   <a href="http://docs.racket-lang.org/quick/index.html">PLT</a>, or
   <a href="http://cs.gettysburg.edu/~tneller/cs341/scheme-intro/index.html">Neller</a>),
   or the
      <a
   href="http://www.schemers.org/Documents/Standards/R5RS/HTML">reference
   manual</a>.

